# 01 — `midas_params`: validate, inspect, rings, discover, wizard

`midas_params` is the parameter-file registry, validator, and wizard
for the MIDAS FF/NF/PF/RI pipelines. This notebook is a fast
end-to-end tour of every public capability on a **self-generated**
sample param file — no network, no external data, runs in a couple
of seconds on CPU.

By the end you will have:

1. Written a small FF `paramstest.txt`.
2. **Validated** it and rendered a human-readable report.
3. Caught a deliberately-broken file with the cross-field rules.
4. **Inspected** the registry (how many keys per pipeline path).
5. Enumerated the visible **Bragg rings** for the crystal + detector.
6. **Discovered** parameters from a raw-frame filename and from a
   calibration file.
7. Run the non-interactive **wizard** to assemble + write a param file.


In [1]:
import tempfile, os
from pathlib import Path

import midas_params
from midas_params import Path as MPath   # the pipeline-path enum (FF/NF/PF/RI)

print("midas_params version:", midas_params.__version__)
# Scratch workspace (self-contained, cleaned up by the OS).
WORK = Path(tempfile.mkdtemp(prefix="midas_params_nb_"))
print("workspace:", WORK)


midas_params version: 0.3.0
workspace: /var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/midas_params_nb_cy4fojos


## 1. A small, valid FF parameter file

We write a minimal-but-coherent FF `paramstest.txt`. Geometry is a
1 m sample-detector distance, 2048², 200 µm pixels, Au (a = 4.08 Å,
FCC, space group 225), four rings, a 360° ω scan in 0.25° steps,
plus the indexing tolerances FF requires.

We also drop a few empty placeholder frames in the workspace so the
later discovery step has a real `RawFolder` to scan.


In [2]:
# Empty placeholder frames (discover_from_file scans the directory).
for n in (1, 2, 3):
    (WORK / f"Au_sample_{n:06d}.ge3").write_bytes(b"")

# RawFolder points at the workspace so the validator's directory check
# passes; OmegaRange matches the scanned [0, 360] range.
SAMPLE = f'''\
Wavelength 0.172979
Lsd 1000000.0
px 200.0
BC 1024 1024
NrPixels 2048
SpaceGroup 225
LatticeConstant 4.08 4.08 4.08 90 90 90
RhoD 200000
RingThresh 1 100
RingThresh 2 100
RingThresh 3 100
RingThresh 4 100
OmegaStart 0
OmegaStep 0.25
OmegaRange 0 360
StartNr 1
EndNr 1440
RawFolder {WORK}
FileStem Au_sample
Ext .ge3
OverAllRingToIndex 1
Completeness 0.6
StepSizeOrient 0.2
StepSizePos 100
tx 0
ty 0
tz 0
'''
param_fn = WORK / "paramstest.txt"
param_fn.write_text(SAMPLE)
print(param_fn.read_text())


Wavelength 0.172979
Lsd 1000000.0
px 200.0
BC 1024 1024
NrPixels 2048
SpaceGroup 225
LatticeConstant 4.08 4.08 4.08 90 90 90
RhoD 200000
RingThresh 1 100
RingThresh 2 100
RingThresh 3 100
RingThresh 4 100
OmegaStart 0
OmegaStep 0.25
OmegaRange 0 360
StartNr 1
EndNr 1440
RawFolder /var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/midas_params_nb_cy4fojos
FileStem Au_sample
Ext .ge3
OverAllRingToIndex 1
Completeness 0.6
StepSizeOrient 0.2
StepSizePos 100
tx 0
ty 0
tz 0



## 2. Validate it

`validate(path_to_file, pipeline_path)` returns a `ValidationReport`.
`format_report` renders it for humans.


In [3]:
from midas_params.validator import validate, format_report

report = validate(str(param_fn), MPath.FF)
print("ok:", report.ok)
print("n errors:", len(report.errors), " n warnings:", len(report.warnings))
print()
print(format_report(report, use_color=False))


ok: True
n errors: 0  n warnings: 0

/var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/midas_params_nb_cy4fojos/paramstest.txt  [FF]
  0 error(s), 0 warning(s), 1 info

  INFO :6  [SpaceGroup] SpaceGroup=225 is the parser default (Fm-3m, Cu/Au/Ni). Confirm this matches your sample.
          rule: space_group_default_smell


## 3. Catch a broken file

The validator runs per-key validators and cross-field consistency
rules. Here we feed it an inconsistent ω-direction (`OmegaStep` is
positive while the scan goes from 180 → -180) and a `StartNr > EndNr`.


In [4]:
BROKEN = '''\
Wavelength 0.172979
Lsd 1000000.0
px 200.0
SpaceGroup 225
LatticeConstant 4.08 4.08 4.08 90 90 90
RingThresh 1 100
OmegaStart 180
OmegaEnd -180
OmegaStep 0.25
StartNr 100
EndNr 50
'''
broken_fn = WORK / "broken.txt"
broken_fn.write_text(BROKEN)

rep = validate(str(broken_fn), MPath.FF)
print("ok:", rep.ok)
for issue in rep.errors:
    print(f"  [error] rule={issue.rule!r}: {issue.message}")


ok: False
  [error] rule='required_key_missing': Required key 'RawFolder' is missing for FF pipeline.
  [error] rule='required_key_missing': Required key 'FileStem' is missing for FF pipeline.
  [error] rule='required_key_missing': Required key 'Ext' is missing for FF pipeline.
  [error] rule='required_key_missing': Required key 'BC' is missing for FF pipeline.
  [error] rule='required_key_missing': Required key 'OmegaRange' is missing for FF pipeline.
  [error] rule='required_key_missing': Required key 'OverAllRingToIndex' is missing for FF pipeline.
  [error] rule='required_key_missing': Required key 'Completeness' is missing for FF pipeline.
  [error] rule='required_key_missing': Required key 'StepSizeOrient' is missing for FF pipeline.
  [error] rule='required_key_missing': Required key 'StepSizePos' is missing for FF pipeline.
  [error] rule='nrpixels_either_or': Detector size is not set. Provide either NrPixels (for a square detector) or both NrPixelsY and NrPixelsZ.
  [error] ru

## 4. Inspect the registry

The registry (`PARAMS`) is the single source of truth — every key is
a JSON-serializable `ParamSpec`. `for_path` / `required_for` scope it
to a pipeline.


In [5]:
from midas_params import PARAMS, for_path, required_for

print("total registered keys:", len(PARAMS))
for p in (MPath.FF, MPath.NF, MPath.PF, MPath.RI):
    n_all = len(for_path(p))
    n_req = len(required_for(p))
    print(f"  {p.name:3s}:  {n_all:3d} keys applicable, {n_req:2d} required")

# Peek at one spec.
spec = {s.name: s for s in PARAMS}["Wavelength"]
print()
print("Wavelength spec:")
print("  type        :", spec.type.name)
print("  units       :", spec.units)
print("  typical     :", spec.typical)
print("  description :", spec.description[:80] if spec.description else None)


total registered keys: 281
  FF :  206 keys applicable, 19 required
  NF :  116 keys applicable, 23 required
  PF :  207 keys applicable, 20 required
  RI :   91 keys applicable, 10 required

Wavelength spec:
  type        : FLOAT
  units       : Å
  typical     : None
  description : X-ray wavelength.


## 5. Enumerate visible Bragg rings

`enumerate_rings` projects the crystal's allowed reflections onto the
detector for the given geometry, sorted by 2θ. `recommend_rings`
picks a sensible subset; `format_ring_table` renders it.


In [6]:
from midas_params.rings import enumerate_rings, recommend_rings, format_ring_table

rings = enumerate_rings(
    wavelength=0.172979,
    lsd_um=1_000_000.0,
    lattice=[4.08, 4.08, 4.08, 90, 90, 90],
    space_group=225,
    rho_d_um=200_000.0,
    nr_pixels_y=2048,
    px_um=200.0,
    max_rings=8,
)
print(format_ring_table(rings, use_color=False))
print()
print("recommended ring numbers:", recommend_rings(rings))


  Ring  (h k l)         d (Å)    2θ (°)    radius (mm)
  ----  ---------       ------   -------   -----------
    1   ( 1  1  1)       2.356     4.208      73.6    ✓
    2   ( 2  0  0)       2.040     4.860      85.0    ✓
    3   ( 2  2  0)       1.442     6.875     120.6    ✓
    4   ( 3  1  1)       1.230     8.063     141.7    ✓
    5   ( 2  2  2)       1.178     8.422     148.1    ✓
    6   ( 4  0  0)       1.020     9.728     171.4    ✓
    7   ( 3  3  1)       0.936    10.604     187.2    ✓
    8   ( 4  2  0)       0.912    10.880     192.2    ✓

recommended ring numbers: [1, 2, 3, 4, 5, 6]


## 6. Discover parameters automatically

`discover_from_file` parses a raw-frame file's **name** into
`RawFolder` / `FileStem` / `Padding` / `Ext` / `StartNr` (and scans
the directory for the frame-number range). `discover_from_calibration_file`
reads an existing MIDAS text param file. `merge` combines sources
with priority (earlier wins).


In [7]:
from midas_params import (
    discover_from_file, discover_from_calibration_file, merge,
)

# discover_from_file inspects an on-disk frame (the placeholders we
# created in step 1) and range-scans the directory.
d_name = discover_from_file(str(WORK / "Au_sample_000001.ge3"))
print("from filename:", d_name.extracted)
print()

# From the calibration param file we wrote in step 1.
d_cal = discover_from_calibration_file(str(param_fn))
print("keys discovered from calibration file:", sorted(d_cal.extracted))
print()

# Merge: filename info wins for file-layout keys, calibration for geometry.
seeded = merge(d_name, d_cal)
print("merged FileStem :", seeded.extracted.get("FileStem"))
print("merged Lsd      :", seeded.extracted.get("Lsd"))
print("merged Wavelength:", seeded.extracted.get("Wavelength"))


from filename: {'RawFolder': '/private/var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/midas_params_nb_cy4fojos', 'FileStem': 'Au_sample', 'Padding': 6, 'Ext': '.ge3', 'StartNr': 1, 'EndNr': 3}

keys discovered from calibration file: ['BC', 'Completeness', 'EndNr', 'Ext', 'FileStem', 'LatticeConstant', 'Lsd', 'NrPixels', 'OmegaRange', 'OmegaStart', 'OmegaStep', 'OverAllRingToIndex', 'RawFolder', 'RhoD', 'RingThresh', 'SpaceGroup', 'StartNr', 'StepSizeOrient', 'StepSizePos', 'Wavelength', 'px', 'tx', 'ty', 'tz']

merged FileStem : Au_sample
merged Lsd      : [1000000.0]
merged Wavelength: 0.172979


## 7. Build a param file with the (non-interactive) wizard

`run_wizard(..., non_interactive=True)` seeds from an existing param
file + the dataset frame we just created, fills any remaining
required keys from registry defaults, writes the file, and validates
it. Returns an exit code (0 = the written file validates clean).


In [8]:
from midas_params.wizard import run_wizard

out_fn = WORK / "wizard_out.txt"
code = run_wizard(
    path=MPath.FF,
    output=str(out_fn),
    from_existing=str(param_fn),
    dataset_file=str(WORK / "Au_sample_000001.ge3"),
    non_interactive=True,
)
print("wizard exit code:", code, "(0 = wrote a file that validates clean)")
print()
print(out_fn.read_text())



Post-write validation:
/var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/midas_params_nb_cy4fojos/wizard_out.txt  [FF]
  0 error(s), 0 warning(s), 1 info

  INFO :27  [SpaceGroup] SpaceGroup=225 is the parser default (Fm-3m, Cu/Au/Ni). Confirm this matches your sample.
          rule: space_group_default_smell
wizard exit code: 0 (0 = wrote a file that validates clean)

# MIDAS parameter file — FF pipeline
# Generated by midas-params wizard


# Data source
RawFolder /var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/midas_params_nb_cy4fojos
FileStem Au_sample
Ext .ge3
Padding 6
StartNr 1
EndNr 1440

# Detector geometry
NrPixels 2048
NrPixelsY 2048
NrPixelsZ 2048
px 200.0
Lsd 1000000.0
BC 1024.0 1024.0
tx 0.0
ty 0.0
tz 0.0
RhoD 200000.0

# Crystallography
LatticeConstant 4.08 4.08 4.08 90.0 90.0 90.0
SpaceGroup 225
Wavelength 0.172979

# Omega scan
OmegaStart 0.0
OmegaEnd 360.0
OmegaStep 0.25
OmegaRange 0.0 360.0

# Ring selection
RingThresh 1 100
RingThresh 2 100
RingThresh 3 100
RingThr

## Recap

| Capability | Entry point |
|---|---|
| Validate a param file | `midas_params.validator.validate` + `format_report` |
| Inspect the registry | `midas_params.PARAMS`, `for_path`, `required_for` |
| Enumerate Bragg rings | `midas_params.rings.enumerate_rings` / `recommend_rings` |
| Discover from filename / calibration | `discover_from_file`, `discover_from_calibration_file`, `merge` |
| Build a file end-to-end | `midas_params.wizard.run_wizard(..., non_interactive=True)` |

The same operations are available from the CLI: `midas-params
validate|inspect|rings|wizard|diagnose`. See the package
[README](../README.md).
